# 02 — Candle Primitives

## Goal
Compute the per-candle features that every downstream step depends on.

## Equations

**Range** — the full price span of a candle:
$$\text{range} = \text{high} - \text{low}$$

**Body** — the absolute distance between open and close:
$$\text{body} = |\text{close} - \text{open}|$$

**Body Ratio** — fraction of the range that is "real move" vs shadow:
$$\text{body\_ratio} = \frac{\text{body}}{\text{range}}$$

- $\text{body\_ratio} \approx 1.0$ → strong directional candle (almost no wicks)
- $\text{body\_ratio} \approx 0.0$ → doji or wide-shadow candle

## Classification Rules

| Flag | Condition | Meaning |
|------|-----------|---------|
| `is_base` | $\text{body\_ratio} \leq 0.50$ | Indecisive / consolidation candle |
| `is_doji` | $\text{body\_ratio} \leq 0.10$ | Near-zero body (perfect indecision) |
| `is_bullish` | $\text{close} > \text{open}$ | Buyers won the session |

## Visualization
The chart below shows candles color-coded by type:
- **Teal** = bullish, **Red** = bearish
- **Grey** = base candle ($\text{body\_ratio} \leq 0.50$)
- **Gold** = doji ($\text{body\_ratio} \leq 0.10$)

The lower panel plots `body_ratio` per candle — the dashed line marks the base threshold at 0.50.


## 1. Load the raw data
Read the OHLC fixtures and define the single threshold constant we need in this notebook.

`BASE_BODY_RATIO_MAX` is the line in the sand: a candle whose body covers **half or less** of its full range is "indecisive" enough to potentially be part of a base.


In [1]:
import pandas as pd

BASE_BODY_RATIO_MAX: float = 0.50   # a candle is "base-like" if body / range ≤ 0.50

df = pd.read_csv("../fixtures.csv", index_col=0, parse_dates=True)
print(f"Loaded {len(df)} candles")
df.head()


Loaded 98 candles


,open,high,low,close,volume
Date,,,,,
2024-01-01 00:00:00+00:00,100.0,101.1,99.5,100.6,1000000
2024-01-08 00:00:00+00:00,100.6,101.1,99.6,100.1,1000000
2024-01-15 00:00:00+00:00,100.1,101.2,99.6,100.7,1000000
2024-01-22 00:00:00+00:00,100.7,101.2,99.7,100.2,1000000
2024-01-29 00:00:00+00:00,100.2,101.3,99.7,100.8,1000000


## 2. Compute `range` and `body`
These are the two raw building blocks. Both come straight from the OHLC values — no smoothing, no lookback, just simple arithmetic on the same row.

- `range = high − low` → the full vertical span of the candle (wicks included)
- `body = |close − open|` → the absolute size of the colored rectangle

We use the absolute value for `body` because we only care about the **magnitude** here. Direction (bullish vs bearish) is captured separately below.


In [2]:
df["range"] = df["high"] - df["low"]
df["body"]  = (df["close"] - df["open"]).abs()

df[["open", "high", "low", "close", "range", "body"]].head()


,open,high,low,close,range,body
Date,,,,,,
2024-01-01 00:00:00+00:00,100.0,101.1,99.5,100.6,1.6,0.6
2024-01-08 00:00:00+00:00,100.6,101.1,99.6,100.1,1.5,0.5
2024-01-15 00:00:00+00:00,100.1,101.2,99.6,100.7,1.6,0.6
2024-01-22 00:00:00+00:00,100.7,101.2,99.7,100.2,1.5,0.5
2024-01-29 00:00:00+00:00,100.2,101.3,99.7,100.8,1.6,0.6


## 3. The key ratio: `body / range`
Now divide body by range. This single number tells you **what kind of candle** you're looking at, independent of price scale.

$$\text{body\_ratio} = \frac{|\text{close} - \text{open}|}{\text{high} - \text{low}}$$

Interpreting the value:
| body_ratio | Shape                                          | Meaning |
|------------|------------------------------------------------|---------|
| ≈ 1.00     | Big body, tiny wicks (marubozu-like)           | Strong, decisive directional move |
| ≈ 0.70     | Body dominates, small wicks                    | Still directional |
| ≈ 0.50     | Body and wicks roughly equal                   | Borderline — start of indecision |
| ≈ 0.20     | Small body, long wicks                         | Buyers and sellers fighting |
| ≈ 0.00     | Almost no body (doji)                          | Perfect indecision |

This ratio is the heart of base detection in NB04.


In [3]:
df["body-ratio"] = df["body"] / df["range"]

df[["body", "range", "body-ratio"]].head(10)


,body,range,body-ratio
Date,,,
2024-01-01 00:00:00+00:00,0.6,1.6,0.375000
2024-01-08 00:00:00+00:00,0.5,1.5,0.333333
2024-01-15 00:00:00+00:00,0.6,1.6,0.375000
2024-01-22 00:00:00+00:00,0.5,1.5,0.333333
2024-01-29 00:00:00+00:00,0.6,1.6,0.375000
2024-02-05 00:00:00+00:00,0.5,1.5,0.333333
2024-02-12 00:00:00+00:00,0.6,1.6,0.375000
2024-02-19 00:00:00+00:00,0.5,1.5,0.333333
2024-02-26 00:00:00+00:00,0.6,1.6,0.375000


## 4. Boolean flags
Turn the ratio into hard yes/no flags that downstream notebooks can filter on directly.

- `is_base` — body ≤ half the range → candidate for a base cluster
- `is_doji` — body ≤ 10% of the range → extreme indecision (perfect base candle)
- `is_bullish` — `close > open` → directional flag (only meaningful for non-base candles, but cheap to compute always)

Note that every doji is also a base (a doji has the smallest body ratio possible), but not every base is a doji.


In [4]:
df["is_base"]    = df["body-ratio"] <= BASE_BODY_RATIO_MAX
df["is_doji"]    = df["body-ratio"] <= 0.10
df["is_bullish"] = df["close"] > df["open"]

# how many candles fall into each bucket
print("is_base   :", df["is_base"].sum(),    "/", len(df))
print("is_doji   :", df["is_doji"].sum(),    "/", len(df))
print("is_bullish:", df["is_bullish"].sum(), "/", len(df))


is_base   : 64 / 98
is_doji   : 4 / 98
is_bullish: 61 / 98


## 5. Save the enriched dataset
The next notebooks need these columns, so we persist them to `fixtures_enhanced.csv`.

From here on, every notebook loads `fixtures_enhanced.csv` instead of the raw `fixtures.csv`.


In [5]:
df.to_csv("../fixtures_enhanced.csv")
print("Saved to ../fixtures_enhanced.csv")
df.head()


Saved to ../fixtures_enhanced.csv


,open,high,low,close,volume,range,body,body-ratio,is_base,is_doji,is_bullish
Date,,,,,,,,,,,
2024-01-01 00:00:00+00:00,100.0,101.1,99.5,100.6,1000000,1.6,0.6,0.375000,True,False,True
2024-01-08 00:00:00+00:00,100.6,101.1,99.6,100.1,1000000,1.5,0.5,0.333333,True,False,False
2024-01-15 00:00:00+00:00,100.1,101.2,99.6,100.7,1000000,1.6,0.6,0.375000,True,False,True
2024-01-22 00:00:00+00:00,100.7,101.2,99.7,100.2,1000000,1.5,0.5,0.333333,True,False,False
2024-01-29 00:00:00+00:00,100.2,101.3,99.7,100.8,1000000,1.6,0.6,0.375000,True,False,True


## 6. Visualization — color coding the candles
Now we plot the same candles, but each one is tinted by **what kind of candle it is**, so the base/doji structure jumps out at a glance.

The chart has two stacked panels sharing the same time axis:

- **Top panel** — candlesticks with custom colors
- **Bottom panel** — `body-ratio` as a bar chart, with the 0.50 threshold drawn as a dashed line

The color palette (dark TradingView-inspired theme):
| Color | Meaning |
|-------|---------|
| Teal `#26a69a` | Bullish, non-base |
| Red `#ef5350`  | Bearish, non-base |
| Grey `#b0bec5` | Base candle (body_ratio ≤ 0.50) |
| Gold `#ffd700` | Doji (body_ratio ≤ 0.10) — overrides base |

The bar in the bottom panel uses the **same color** as the candle above it, so you can scan vertically and immediately see "this short bar = doji, that tall bar = strong directional candle".


In [6]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "notebook_connected"

# ── dark theme palette (used in every notebook) ──────────────────────────
COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
COLOR_BASE = "#b0bec5"
COLOR_DOJI = "#ffd700"
BG         = "#131722"
GRID       = "#1e222d"

def candle_color(row):
    """Pick a color by candle type. Doji wins over base, base wins over direction."""
    if row["is_doji"]: return COLOR_DOJI
    if row["is_base"]: return COLOR_BASE
    return COLOR_BULL if row["is_bullish"] else COLOR_BEAR

colors = df.apply(candle_color, axis=1)


## 7. Build the two-panel figure
Use `make_subplots` so both panels share an x-axis. We give the top (candles) 70% of vertical space and the bottom (ratio bars) 30% — the candle pattern is the main event; the bar chart is a sanity check.


In [7]:
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.7, 0.3],
    vertical_spacing=0.03,
)

# top panel — candlestick chart
fig.add_trace(go.Candlestick(
    x=df.index,
    open=df["open"], high=df["high"],
    low=df["low"],   close=df["close"],
    increasing_line_color=COLOR_BULL,
    decreasing_line_color=COLOR_BEAR,
    name="Price",
), row=1, col=1)

# bottom panel — body-ratio bars colored by candle type
fig.add_trace(go.Bar(
    x=df.index,
    y=df["body-ratio"],
    marker_color=colors,
    name="Body Ratio",
    showlegend=False,
), row=2, col=1)

# threshold reference line at 0.50
fig.add_hline(
    y=BASE_BODY_RATIO_MAX,
    line_dash="dot", line_color="#888", line_width=1,
    annotation_text=f"base ≤ {BASE_BODY_RATIO_MAX}",
    annotation_font_color="#888",
    annotation_position="bottom right",
    row=2, col=1,
)


## 8. Apply the dark theme and render
Last step — style both panels with the dark background, grid color, and labels. This same styling is reused across every chart in the series, so it's worth understanding once.


In [8]:
fig.update_layout(
    title="Candle Primitives — base, doji & body ratio",
    height=600,
    plot_bgcolor=BG,
    paper_bgcolor=BG,
    font_color="white",
    xaxis_rangeslider_visible=False,
    legend=dict(orientation="h", y=1.04),
    bargap=0.15,
)

# style both x-axes and y-axes the same way
for axis in ["xaxis", "xaxis2", "yaxis", "yaxis2"]:
    fig.update_layout(**{axis: dict(
        gridcolor=GRID,
        showgrid=True,
        zeroline=False,
    )})

fig.update_yaxes(title_text="Price",      row=1, col=1)
fig.update_yaxes(title_text="Body Ratio", row=2, col=1, range=[0, 1.1])

fig.show()
